<a href="https://colab.research.google.com/github/fboldt/aulasml/blob/master/aula04a_%C3%A1rvore_de_decis%C3%A3o_com_atributos_discretos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ucimlrepo -q

In [8]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
car_evaluation = fetch_ucirepo(id=19)

# data (as pandas dataframes)
X = car_evaluation.data.features.to_numpy()
y = car_evaluation.data.targets.to_numpy().reshape(-1,)

# metadata
print(car_evaluation.metadata)

# variable information
print(car_evaluation.variables)

{'uci_id': 19, 'name': 'Car Evaluation', 'repository_url': 'https://archive.ics.uci.edu/dataset/19/car+evaluation', 'data_url': 'https://archive.ics.uci.edu/static/public/19/data.csv', 'abstract': 'Derived from simple hierarchical decision model, this database may be useful for testing constructive induction and structure discovery methods.', 'area': 'Other', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 1728, 'num_features': 6, 'feature_types': ['Categorical'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1988, 'last_updated': 'Thu Aug 10 2023', 'dataset_doi': '10.24432/C5JP48', 'creators': ['Marko Bohanec'], 'intro_paper': {'ID': 249, 'type': 'NATIVE', 'title': 'Knowledge acquisition and explanation for multi-attribute decision making', 'authors': 'M. Bohanec, V. Rajkovič', 'venue': '8th Intl Workshop on Expert Systems and their Applications, 

In [9]:
print(X.shape)
print(y.shape)

(1728, 6)
(1728,)


In [12]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier()
# # ERRRO
# model.fit(X, y)
# y_pred = model.predict(X)

In [14]:
for i in range(X.shape[1]):
  values = set(X[:,i])
  print(f"{i}. {car_evaluation.variables['name'][i]}:\t{values}")

0. buying:	{'vhigh', 'low', 'med', 'high'}
1. maint:	{'vhigh', 'low', 'med', 'high'}
2. doors:	{'5more', '3', '4', '2'}
3. persons:	{'more', '4', '2'}
4. lug_boot:	{'small', 'med', 'big'}
5. safety:	{'high', 'low', 'med'}


In [15]:
print(set(y))

{'good', 'vgood', 'acc', 'unacc'}


In [17]:
for label in set(y):
  print(f"{label}:\t{100*sum(y==label)/len(y):.4}%")

good:	3.993%
vgood:	3.762%
acc:	22.22%
unacc:	70.02%


In [21]:
import numpy as np
labels, counts = np.unique(y, return_counts=True)
print(labels)
print(counts)

['acc' 'good' 'unacc' 'vgood']
[ 384   69 1210   65]


In [53]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score
import numpy as np

def most_frequent(arr):
  labels, counts = np.unique(arr, return_counts=True)
  return labels[np.argmax(counts)]

class ZeroR(BaseEstimator, ClassifierMixin):
  def fit(self, X, y):
    self.answer = most_frequent(y)
    return self
  def predict(self, X):
    return [self.answer]*X.shape[0]

model = ZeroR()
model.fit(X, y)
y_pred = model.predict(X)
print(accuracy_score(y, y_pred))

0.7002314814814815


In [54]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    stratify=y,
                                                    shuffle=True,
                                                    random_state=42)

model = ZeroR()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.6994219653179191


In [55]:
feature = np.random.randint(X.shape[1])
print(feature)
value = np.random.choice(X[:,feature])
print(value)
value = np.random.choice(list(set(X[:,feature])))
print(value)

0
med
low


In [70]:
import numpy as np

class DecisionTree(BaseEstimator, ClassifierMixin):
  def fit(self, X, y):
    self.feature = np.random.randint(X.shape[1])
    self.value = np.random.choice(X[:,self.feature])
    equals = X[:,self.feature] == self.value
    if sum(equals) == 0 or sum(~equals) == 0:
      self.answer = most_frequent(y)
    else:
      self.equals = DecisionTree().fit(X[equals], y[equals])
      self.not_equals = DecisionTree().fit(X[~equals], y[~equals])
    return self
  def predict(self, X):
    if hasattr(self, 'answer'):
      return [self.answer]*X.shape[0]
    else:
      equals = X[:,self.feature] == self.value
      return np.where(equals, self.equals.predict(X), self.not_equals.predict(X))

model = DecisionTree()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.7225433526011561


In [84]:
labels, counts = np.unique(y_train, return_counts=True)
print(labels)
print(counts)
print(sum(counts))
p = counts/sum(counts)
print(p)
print(p**2)
print("Gini:", 1-sum(p**2))

['acc' 'good' 'unacc' 'vgood']
[307  55 968  52]
1382
[0.22214182 0.0397974  0.70043415 0.03762663]
[0.04934699 0.00158383 0.490608   0.00141576]
Gini: 0.4570454112310227


In [85]:
def gini(y):
  labels, counts = np.unique(y, return_counts=True)
  p = counts/sum(counts)
  return 1-sum(p**2)

print(gini(y_train))

0.4570454112310227


In [86]:
print(gini(np.ones(100)))

0.0


In [87]:
print(gini(np.arange(100)))

0.99


In [90]:
def impurity_value(x, y, value, impurity_function):
  equals = x == value
  equals_impurity = impurity_function(y[equals])
  not_equals_impurity = impurity_function(y[~equals])
  return (sum(equals)*equals_impurity + sum(~equals)*not_equals_impurity)/len(y)

print(impurity_value(X_train[:,0], y_train, "vhigh", gini))

0.4482135562918824


In [100]:
def best_split(x, y, impurity_function):
  best_value = None
  best_impurity = np.inf
  for value in set(x):
    impurity = impurity_value(x, y, value, impurity_function)
    if impurity < best_impurity:
      best_impurity = impurity
      best_value = value
  return best_value, best_impurity

print(best_split(X_train[:,3], y_train, gini))

('2', np.float64(0.387305111020565))


In [101]:
def best_feature(X, y, impurity_function):
  best_feature = None
  best_value = None
  best_impurity = np.inf
  for feature in range(X.shape[1]):
    value, impurity = best_split(X[:,feature], y, impurity_function)
    if impurity < best_impurity:
      best_feature = feature
      best_value = value
      best_impurity = impurity
  return best_feature, best_value, best_impurity

print(best_feature(X_train, y_train, gini))

(5, 'low', np.float64(0.38499511557696947))


In [102]:
class DecisionTree(BaseEstimator, ClassifierMixin):
  def fit(self, X, y):
    self.feature, self.value, self.impurity = best_feature(X, y, gini)
    equals = X[:,self.feature] == self.value
    if sum(equals) == 0 or sum(~equals) == 0:
      self.answer = most_frequent(y)
    else:
      self.equals = DecisionTree().fit(X[equals], y[equals])
      self.not_equals = DecisionTree().fit(X[~equals], y[~equals])
    return self
  def predict(self, X):
    if hasattr(self, 'answer'):
      return [self.answer]*X.shape[0]
    else:
      equals = X[:,self.feature] == self.value
      return np.where(equals, self.equals.predict(X), self.not_equals.predict(X))

model = DecisionTree()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9682080924855492


In [103]:
model = DecisionTree()
model.fit(X, y)
y_pred = model.predict(X)
print(accuracy_score(y, y_pred))

1.0


In [117]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

model = DecisionTree()
scores = cross_val_score(model, X, y, cv=StratifiedKFold(5))
print(scores)
print(np.mean(scores))

[0.62716763 0.73121387 0.74855491 0.75362319 0.8057971 ]
0.7332713412080086


In [118]:
class DecisionTree(BaseEstimator, ClassifierMixin):
  def __init__(self, max_depth=1000):
    self.max_depth = max_depth

  def fit(self, X, y):
    self.feature, self.value, self.impurity = best_feature(X, y, gini)
    equals = X[:,self.feature] == self.value
    if sum(equals) == 0 or sum(~equals) == 0 or self.max_depth == 0:
      self.answer = most_frequent(y)
    else:
      self.equals = DecisionTree(self.max_depth-1).fit(X[equals], y[equals])
      self.not_equals = DecisionTree(self.max_depth-1).fit(X[~equals], y[~equals])
    return self
  def predict(self, X):
    if hasattr(self, 'answer'):
      return [self.answer]*X.shape[0]
    else:
      equals = X[:,self.feature] == self.value
      return np.where(equals, self.equals.predict(X), self.not_equals.predict(X))

model = DecisionTree()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9682080924855492


In [119]:
model = DecisionTree(1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.6994219653179191


In [128]:
model = DecisionTree(5)
scores = cross_val_score(model, X, y, cv=StratifiedKFold(5))
print(scores)
print(np.mean(scores))

[0.77456647 0.74855491 0.64739884 0.73333333 0.76521739]
0.7338141911703108


In [140]:
class DecisionTree(BaseEstimator, ClassifierMixin):
  def __init__(self, max_depth=1000, min_sample_split=1):
    self.max_depth = max_depth
    self.min_sample_split = min_sample_split

  def fit(self, X, y):
    self.feature, self.value, self.impurity = best_feature(X, y, gini)
    equals = X[:,self.feature] == self.value
    if sum(equals) < self.min_sample_split or sum(~equals) < self.min_sample_split or self.max_depth == 0:
      self.answer = most_frequent(y)
    else:
      self.equals = DecisionTree(self.max_depth-1, self.min_sample_split).fit(X[equals], y[equals])
      self.not_equals = DecisionTree(self.max_depth-1, self.min_sample_split).fit(X[~equals], y[~equals])
    return self
  def predict(self, X):
    if hasattr(self, 'answer'):
      return [self.answer]*X.shape[0]
    else:
      equals = X[:,self.feature] == self.value
      return np.where(equals, self.equals.predict(X), self.not_equals.predict(X))

model = DecisionTree()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9682080924855492


In [142]:
model = DecisionTree(1000, 10)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9508670520231214


In [148]:
model = DecisionTree(1000, 10)
scores = cross_val_score(model, X, y, cv=StratifiedKFold(5))
print(scores)
print(np.mean(scores))

[0.62427746 0.72543353 0.64739884 0.79710145 0.76521739]
0.7118857334338611


In [149]:
!pip install optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 10.8 MB/s eta 0:00:00


In [154]:
import optuna
from sklearn.model_selection import cross_val_score, KFold

def objective(trial):
  max_depth = trial.suggest_int('max_depth', 1, 50)
  min_sample_split = trial.suggest_int('min_sample_split', 1, 20)

  model = DecisionTree(max_depth=max_depth, min_sample_split=min_sample_split)
  scores = cross_val_score(model, X_train, y_train, cv=KFold(5, shuffle=True))
  return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50) # You can adjust n_trials as needed

print("Best trial:")
print(f"  Value: {study.best_value}")
print(f"  Params: {study.best_params}")

print("Final evaluation:")
model = DecisionTree(**study.best_params)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

[I 2026-09-14 23:49:42,326] A new study created in memory with name: no-name-722df087-8a03-4f7b-a7cc-6bf2b5582545
[I 2026-09-14 23:49:43,568] Trial 0 finished with value: 0.8965259247632501 and parameters: {'max_depth': 22, 'min_sample_split': 15}. Best is trial 0 with value: 0.8965259247632501.
[I 2026-09-14 23:49:44,852] Trial 1 finished with value: 0.9182205828493697 and parameters: {'max_depth': 13, 'min_sample_split': 11}. Best is trial 1 with value: 0.9182205828493697.
[I 2026-09-14 23:49:46,035] Trial 2 finished with value: 0.9030241197090986 and parameters: {'max_depth': 49, 'min_sample_split': 13}. Best is trial 1 with value: 0.9182205828493697.
[I 2026-09-14 23:49:47,396] Trial 3 finished with value: 0.9240229163396642 and parameters: {'max_depth': 16, 'min_sample_split': 9}. Best is trial 3 with value: 0.9240229163396642.
[I 2026-09-14 23:49:48,939] Trial 4 finished with value: 0.919672474232198 and parameters: {'max_depth': 13, 'min_sample_split': 6}. Best is trial 3 with v

Best trial:
  Value: 0.9681525663161199
  Params: {'max_depth': 30, 'min_sample_split': 1}
Final evaluation:
0.9682080924855492
